<a href="https://colab.research.google.com/github/Muskan624-ai/Bias-detector/blob/main/data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import shutil

In [3]:
df = pd.read_csv('/content/drive/MyDrive/BiasProject/data.csv')
print("--- Sample Data ---")
print(df.head())

df = df.dropna()

--- Sample Data ---
                                    text  label
0   the media always lies about politics      1
1  the government announced a new policy      0
2   this law unfairly targets minorities      1
3         the meeting was held yesterday      0
4            this news channel is biased      1


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

df['text'] = df['text'].apply(clean_text)


In [6]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

print(f"\nData Prepared!")
print(f"Total rows in dataset: {len(df)}")
print(f"Training on: {len(train_texts)} sentences")
print(f"Testing on: {len(test_texts)} sentences")


Data Prepared!
Total rows in dataset: 5400
Training on: 4320 sentences
Testing on: 1080 sentences


In [7]:
print("\nInitializing Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)
print("Tokenization Complete!")


Initializing Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization Complete!


In [9]:
class BiasDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Convert arrays into explicit Torch datasets for the trainer engine
train_dataset = BiasDataset(train_encodings, train_labels)
test_dataset = BiasDataset(test_encodings, test_labels)
print("PyTorch Datasets ready for model input!")

PyTorch Datasets ready for model input!


In [10]:
print("\nLoading Pre-trained DistilBERT Architecture (2 Labels)...")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model moved to operational device: {device}")


Loading Pre-trained DistilBERT Architecture (2 Labels)...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model moved to operational device: cpu


In [11]:
training_args = TrainingArguments(
    output_dir='./results',          # Temporary training checkpoint storage
    num_train_epochs=3,              # How many times the model looks at the whole dataset
    per_device_train_batch_size=16,  # Batch processing constraint
    per_device_eval_batch_size=16,
    warmup_steps=500,                # Gradually increases learning rate
    weight_decay=0.01,               # Structural penalty against overfitting
    eval_strategy="epoch",           # Evaluates error rates at each complete cycle
    logging_steps=10,
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()
print("Model Training Cycle Complete")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.001667,0.001096
2,0.000180,0.000109
3,0.000092,0.000054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Model Training Cycle Complete


In [13]:
print("\nArchiving weights locally inside Colab workspace...")
model.save_pretrained("./bias_model_final")
tokenizer.save_pretrained("./bias_model_final")

# Create a zip archive of the model files
shutil.make_archive("bias_model_final", 'zip', "./bias_model_final")
print("✅ Local zip file generated.")

# Copy the local zip directly to your shared Google Drive folder!
print("⏳ Exporting final model weights zip to Google Drive...")
shutil.copy("bias_model_final.zip", "/content/drive/MyDrive/BiasProject/bias_model_final.zip")
print("The model weights are securely stored on your Google Drive.")


Archiving weights locally inside Colab workspace...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Local zip file generated.
⏳ Exporting final model weights zip to Google Drive...
The model weights are securely stored on your Google Drive.
